# Dataset analysis

In [1]:
import pandas as pd
import sqlite3

In [21]:
df = pd.read_sql_query("SELECT * FROM Reviews", sqlite3.connect(r"archive/database.sqlite"))

In [22]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [23]:
df['Score'].value_counts()

Score
5    363122
4     80655
1     52268
3     42640
2     29769
Name: count, dtype: int64

In [26]:
df_main = df[["Text", "Score"]]
df_main = df.dropna()

df_main.head()

,Text,Score
0,I have bought several of the Vitality canned d...,5
1,Product arrived labeled as Jumbo Salted Peanut...,1
2,This is a confection that has been around a fe...,4
3,If you are looking for the secret ingredient i...,2
4,Great taffy at a great price. There was a wid...,5


In [ ]:
def convert_label(score):

    # Negative
    if score <= 2:
        return 0

    # Positive
    elif score >= 4:
        return 1

    # Neutral
    else:
        return -1


df["labels"] = df["Score"].apply(convert_label)

# Remove neutral reviews
df = df[df["labels"] != -1]

df = df.rename(columns={"Text": "text"})

In [ ]:
negative_df = df[df["labels"] == 0]

positive_df = df[df["labels"] == 1].sample(
    n=len(negative_df),   # balance dataset
    random_state=42
)

df = pd.concat([negative_df, positive_df])

# Shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df["labels"].value_counts())

label
0    82037
1    82037
Name: count, dtype: int64


In [30]:
len(df)

164074

In [31]:
df.to_csv("balanced_reviews.csv", index=False)